## Tool Schema

In [ ]:
# OpenAI uses a slightly different schema format compared to Anthropic.
# Tools are wrapped under a 'function' key with 'type': 'function'.
tools = [
    {
        "type": "function",
        "function": {
            "name": "say_hello",
            "description": "Greeting message for user",
            "parameters": {
                "type": "object",
                "properties": {
                    "name": {
                        "type": "string",
                        "description": "Name of user",
                    },
                },
                "required": ["name"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_current_date",
            "description": "Return current date base on python code",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": [],
            },
        },
    },
]

In [ ]:
from datetime import datetime


def say_hello(name: str):
    return f"Hello {name}"


def get_current_date():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

## Tool Mapping

In [ ]:
mapping_tool_function = {"say_hello": say_hello, "get_current_date": get_current_date}

## Tool Executer Method

In [ ]:
import json


def execute_tool(tool_name, tool_args):
    result = mapping_tool_function[tool_name](**tool_args)

    if result is None:
        result = "The operation completed but didn't return any results."
    elif isinstance(result, list):
        result = ", ".join(result)
    elif isinstance(result, dict):
        result = json.dumps(result, indent=2)
    else:
        result = str(result)

    return result

## Test With ChatBot

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
# OpenAI client reads OPENAI_API_KEY from environment automatically
client = OpenAI()

In [ ]:
def process_query(query):
    messages = [{'role': 'user', 'content': query}]

    response = client.chat.completions.create(
        model='gpt-4o',
        tools=tools,
        messages=messages
    )

    process_flag = True
    while process_flag:
        message = response.choices[0].message
        finish_reason = response.choices[0].finish_reason

        # If no tool calls, just print the text and stop
        if finish_reason == 'stop' or not message.tool_calls:
            print(message.content)
            process_flag = False

        # Handle tool calls
        elif finish_reason == 'tool_calls':
            # Add assistant message (with tool calls) to history
            messages.append(message)

            for tool_call in message.tool_calls:
                tool_name = tool_call.function.name
                tool_args = json.loads(tool_call.function.arguments)
                tool_call_id = tool_call.id

                print(f"Calling tool {tool_name} with args {tool_args}")
                result = execute_tool(tool_name, tool_args)

                # Append tool result to messages
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call_id,
                    "content": result
                })

            # Send updated messages back to model
            response = client.chat.completions.create(
                model='gpt-4o',
                tools=tools,
                messages=messages
            )

            # If next response is just text, print and stop
            next_message = response.choices[0].message
            if response.choices[0].finish_reason == 'stop':
                print(next_message.content)
                process_flag = False


In [ ]:
def chat_loop():
    print("Type your queries or 'q' to exit.")
    while True:
        try:
            query = input("\nQuery: ").strip()
            if query.lower() == "q":
                break
            process_query(query)
            print("\n")
        except Exception as e:
            print(f"\nError: {str(e)}")

In [ ]:
chat_loop()